# Capstone 4 — Coding Agents for a Team · Homework (self-test)

**Goal:** check, without looking at the answers, that you can configure one repository for Claude Code **and** Codex, run both in CI with structured output, and hold agent changes to a verification contract.

- **Brief and theory:** [README.md](README.md) · **Practice:** [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 90–120 minutes
- **Estimated API cost:** **$0**. Everything is offline: no exercise calls a CLI, an SDK or an API.
- **Prerequisites:** the repo venv from [00 — Prerequisites](../../../00-prerequisites/README.md)

### How to use this notebook

1. Answer the quiz in Part A by filling in `answers`, then run the grader.
2. Implement each exercise in Part B (replace `raise NotImplementedError`), then run its check cell. Checks print ✓ or a hint and never stop the notebook.
3. Write your answer to the Part C scenario **before** opening the rubric.
4. Run the scorecard. Solutions are at the very bottom; try everything first.

**Scoring:** quiz (10 questions) + exercises (6). The pass mark is **72%** of the combined score, mirroring the Claude exam's 720/1000.

In [ ]:
import ast
import hashlib
import json
import re
import shlex
import tomllib
from typing import Any

from dotenv import find_dotenv, load_dotenv

try:
    import yaml  # PyYAML is installed with JupyterLab (requirements.txt); not a direct dependency
except ImportError:  # pragma: no cover
    yaml = None
    print("PyYAML is missing: `pip install pyyaml` (it normally ships with JupyterLab). Exercise 2 needs it.")

load_dotenv(find_dotenv())

RESULTS: dict[str, bool] = {f"ex{i}": False for i in range(1, 7)}
QUIZ_SCORE: dict[str, int] = {"correct": 0, "total": 10}


def run_check(name: str, fn) -> None:
    """Run a checker without ever crashing the notebook."""
    try:
        fn()
        RESULTS[name] = True
        print(f"✓ Exercise {name[2:]} passed")
    except NotImplementedError:
        RESULTS[name] = False
        print(f"✗ Exercise {name[2:]}: not implemented yet (replace the NotImplementedError)")
    except AssertionError as e:
        RESULTS[name] = False
        print(f"✗ Exercise {name[2:]}: {e}")
    except Exception as e:  # noqa: BLE001 - a bug in learner code is reported, not raised
        RESULTS[name] = False
        print(f"✗ Exercise {name[2:]} raised {type(e).__name__}: {e}")


print("ready")

In [ ]:
# Helpers the checkers use (you may call them too). They mirror the rules from the practice notebook.

class AgentRunError(RuntimeError):
    """An agent run that did not finish cleanly."""


IMPORT_RE = re.compile(r"(?:(?<=\s)|^)@([^\s`]+)")


def find_imports(text: str) -> list[str]:
    """`@path` imports outside fenced code blocks and inline code spans."""
    found, fenced = [], False
    for line in text.splitlines():
        if line.lstrip().startswith("```"):
            fenced = not fenced
            continue
        if not fenced:
            found += IMPORT_RE.findall(re.sub(r"`[^`]*`", "", line))
    return found


def parse_frontmatter(text: str) -> tuple[dict, str]:
    if not text.startswith("---\n"):
        return {}, text
    end = text.find("\n---", 4)
    if end == -1:
        return {}, text
    meta = yaml.safe_load(text[4:end]) or {}
    return (meta if isinstance(meta, dict) else {}), text[end + 4:].lstrip("\n")


def expand_braces(pattern: str) -> list[str]:
    m = re.search(r"\{([^{}]*)\}", pattern)
    if not m:
        return [pattern]
    return [x for alt in m.group(1).split(",") for x in expand_braces(pattern[: m.start()] + alt + pattern[m.end():])]


def glob_regex(pattern: str) -> str:
    out, i = "", 0
    while i < len(pattern):
        if pattern.startswith("**/", i):
            out, i = out + "(?:.*/)?", i + 3
        elif pattern.startswith("**", i):
            out, i = out + ".*", i + 2
        elif pattern[i] == "*":
            out, i = out + "[^/]*", i + 1
        elif pattern[i] == "?":
            out, i = out + "[^/]", i + 1
        else:
            out, i = out + re.escape(pattern[i]), i + 1
    return out


def glob_match(pattern: str, path: str) -> bool:
    return any(re.fullmatch(glob_regex(p.strip()), path) for p in expand_braces(pattern))


def rule_globs(meta: dict) -> list[str]:
    paths = meta.get("paths", [])
    return [p.strip() for p in paths.split(",")] if isinstance(paths, str) else list(paths)


RULE_RE = re.compile(r"^(\w+)(?:\((.*)\))?$")


def bash_rule_matches(spec: str, command: str) -> bool:
    command = " ".join(command.split())
    if spec.endswith(" *") and spec.count("*") == 1:
        prefix = spec[:-2]
        return command == prefix or command.startswith(prefix + " ")
    return re.fullmatch(".*".join(re.escape(p) for p in spec.split("*")), command) is not None


def claude_decide(tool: str, arg: str, settings: dict) -> str:
    """deny -> ask -> allow, first match wins; 'default' when no rule matches."""
    for decision in ("deny", "ask", "allow"):
        for rule in settings.get("permissions", {}).get(decision, []):
            m = RULE_RE.match(rule)
            if not m or m.group(1) != tool:
                continue
            spec = m.group(2)
            if spec is None:
                return decision
            if tool == "Bash" and bash_rule_matches(spec, arg):
                return decision
            if tool in {"Read", "Edit"} and glob_match(spec[2:] if spec.startswith("./") else spec.lstrip("/"), arg):
                return decision
    return "default"


def load_codex_rules(text: str) -> list[dict]:
    rules = []
    for node in ast.walk(ast.parse(text)):
        if isinstance(node, ast.Call) and getattr(node.func, "id", "") == "prefix_rule":
            kw = {k.arg: ast.literal_eval(k.value) for k in node.keywords}
            kw.setdefault("decision", "allow")
            rules.append(kw)
    return rules


def prefix_matches(pattern: list, argv: list[str]) -> bool:
    return len(argv) >= len(pattern) and all(a in (p if isinstance(p, list) else [p]) for p, a in zip(pattern, argv))


def codex_decide(command: str, rules: list[dict]) -> str | None:
    order = {"allow": 0, "prompt": 1, "forbidden": 2}
    hits = [r["decision"] for r in rules if prefix_matches(r["pattern"], shlex.split(command))]
    return max(hits, key=order.__getitem__) if hits else None


print("helpers ready")

## Part A — Quiz (10 scenario questions)

Pick one letter per question. Each question has exactly one best answer under the **current** Claude Code and Codex docs.

**Q1.** A repo has only `AGENTS.md` (no `CLAUDE.md`), and Claude Code users on a current version follow the team rules. One developer adds a gitignored `CLAUDE.local.md` with personal sandbox URLs. From then on, only that developer's Claude sessions ignore the team rules; Codex is unaffected. What is the best **team-level** fix?

- A) Ask the developer to delete `CLAUDE.local.md` and put the personal URLs in `AGENTS.md`
- B) Commit `.claude/settings.json` that sets **Project instructions** to `claude-md-and-agents-md`
- C) Commit a root `CLAUDE.md` whose first line is `@AGENTS.md`
- D) Rename `AGENTS.md` to `CLAUDE.md`

**Q2.** Payments rules live in `services/payments/AGENTS.md`. Developers start Codex at the repository root and ask it to change `services/payments/refund.py`, and Codex ignores the payments rules. Which explanation and fix are correct?

- A) Codex reads `AGENTS.md` files only from the project root down to its working directory; start Codex in `services/payments/` (or use `--cd`), or route to the payments guide from the root `AGENTS.md`
- B) Codex caches `AGENTS.md` per repository; run `/clear` to reload it
- C) Subdirectories need `AGENTS.override.md`; plain `AGENTS.md` is only read at the root
- D) The root `AGENTS.md` exceeded the 32 KiB cap, so nested files were dropped

**Q3.** The team wants one `release-notes` skill that developers can invoke by name in **both** Claude Code and Codex, maintained in one place. Because it writes to `CHANGELOG.md`, it must never run **implicitly** in Codex. Which layout fits?

- A) `.claude/skills/release-notes/SKILL.md` with `disable-model-invocation: true`, copied to `.codex/skills/release-notes/`
- B) A `## Release notes` section in the root `AGENTS.md`
- C) `~/.agents/skills/release-notes/` installed on every laptop
- D) `.agents/skills/release-notes/` with spec-only `SKILL.md` frontmatter and `agents/openai.yaml` setting `allow_implicit_invocation: false`, plus `.claude/skills/release-notes` as a symlink to that folder

**Q4.** An internal gateway accepts the same key as the Claude API, so a teammate commits this `.mcp.json` entry for a remote HTTP server: `"headers": {"Authorization": "Bearer ${ANTHROPIC_API_KEY}"}`. `ANTHROPIC_API_KEY` is set, yet the server answers 401. What is going on?

- A) `.mcp.json` does not expand variables inside `headers`
- B) Claude Code deliberately reads its own credential variables as empty in a remote server's `url` and `headers`; copy the value into a separately named variable and reference that
- C) The variable needs a default: `${ANTHROPIC_API_KEY:-}`
- D) Expansion only works in `~/.claude.json`; move the server to user scope

**Q5.** A developer's `~/.claude/settings.json` allows `Bash(git push --force origin feature/*)`. The project's `.claude/settings.json` denies `Bash(git push --force *)`. What happens when Claude tries `git push --force origin feature/x`?

- A) Allowed, because the more specific rule wins
- B) Allowed, because user settings override project settings
- C) Denied, because deny rules are evaluated before allow rules regardless of scope or specificity
- D) Claude asks, because the two rules conflict

**Q6.** A CI job runs `codex exec --json --output-schema review.schema.json "..."` in a fresh checkout on an ephemeral runner. The repo's `.codex/config.toml` defines the tracker MCP server with `required = true`, yet the run behaves as if the server did not exist and still succeeds. What is the most likely cause?

- A) `required` only applies to stdio servers
- B) `codex exec` never starts MCP servers
- C) `--output-schema` disables tool use
- D) The checkout is not a trusted project on that runner, so Codex skips the project `.codex/` layer; put CI-critical settings in the runner's own config (or a profile, or `-c` overrides)

**Q7.** Your review bot sometimes cites line numbers that GitHub rejects for inline comments, and some findings quote code that does not exist. Which change fixes both with the least reliance on the model behaving?

- A) Require an `evidence` field (exact text of the cited line) in the schema; post inline only when the line is in the diff and the evidence matches that line; send other valid findings to the review summary
- B) Ask the model to double-check its line numbers before answering
- C) Raise `--max-turns` so the model can re-read files
- D) Turn on extended thinking for the review step

**Q8.** A nightly job lets an agent add missing tests and open a pull request. Which design follows the documented security guidance?

- A) One job with `contents: write`, the API key as a job-level environment variable, and the agent pushing its own branch
- B) The agent job has `contents: read` and the key (behind the action's proxy for Codex) and uploads a patch artifact; a separate job without the key applies the patch and opens the PR
- C) Run the agent with `danger-full-access` so `git push` works inside the sandbox
- D) Commit the key to a `.env` file that only the agent reads

**Q9.** Which task benefits most from starting in plan mode?

- A) Add a missing null check on the line a stack trace points to
- B) Rename a local variable inside one function
- C) Fix a typo in an error message
- D) Add a caching layer for price lookups where an in-process LRU, Redis and HTTP caching are all viable, touching the API, pricing code and deployment config

**Q10.** An agent reports "verified: all 42 tests pass". The baseline had 44 tests; the diff deletes two flaky tests. What should the verification contract and its validator do?

- A) Accept: every remaining test passes
- B) Accept, as long as the PR description explains the flakiness
- C) Reject the `verified` status: the test count fell below the baseline and tests were removed, which the contract forbids. The agent should report the problem and leave flaky-test handling to a human
- D) Restore the tests automatically, re-run, and accept whatever the result is

In [ ]:
# Fill in one letter per question: "A", "B", "C" or "D".
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
}

In [ ]:
ANSWER_KEY = {'q1': '227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e', 'q2': '3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45', 'q3': '14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6', 'q4': '61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa', 'q5': 'e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f', 'q6': '70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9', 'q7': '8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1', 'q8': '39f95f59afaa9522616af26bd34df1e0184030ae8f738e319fc9a1f15b075834', 'q9': 'e58517837d895fe6365a5801be9a66d9d1d39f1d863920a84cbed549cfec1954', 'q10': 'd6ea6e76ddfa07a3b78a8bd000db76a223a8ccc3f38e290cf5a68d95e5494cc5'}  # sha256 of "qid:LETTER", so the answers are not visible


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid, digest in ANSWER_KEY.items():
        letter = (answers.get(qid) or "").strip().upper()
        ok = bool(letter) and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == digest
        correct += ok
        print(f"{qid:>4}: {letter or '-':2} {'✓' if ok else '✗'}")
    QUIZ_SCORE["correct"] = correct
    print(f"Quiz score: {correct}/{len(ANSWER_KEY)} ({correct / len(ANSWER_KEY):.0%})")
    return correct


grade_quiz(answers)

## Part B — Exercises

All six are offline. Each check cell prints ✓ or a hint and never stops the notebook.

### Exercise 1 — Write the shared instruction layer

Return `{"AGENTS.md": ..., "CLAUDE.md": ...}` for a repo used by both agents:

- `AGENTS.md`: mentions the test command `make test`; states that money is integer **cents**; has a `## Code Review Rules` heading; contains **no** `@path` imports (Codex would read them as plain text); at most 120 lines.
- `CLAUDE.md`: the first non-empty line is exactly `@AGENTS.md`; at least one bullet (`- ...`) of its own for Claude-only guidance; **no** bullet that repeats one from `AGENTS.md`, and it does not repeat the test command.

[Theory](README.md#41-instruction-layering-one-source-of-truth)

In [ ]:
def write_instruction_layer() -> dict[str, str]:
    # TODO: return {"AGENTS.md": "...", "CLAUDE.md": "..."}
    raise NotImplementedError

In [ ]:
def _check_ex1():
    files = write_instruction_layer()
    assert isinstance(files, dict) and {"AGENTS.md", "CLAUDE.md"} <= set(files), "return a dict with AGENTS.md and CLAUDE.md"
    agents, claude = files["AGENTS.md"], files["CLAUDE.md"]
    bullets = lambda t: {re.sub(r"\s+", " ", l.strip()[2:]).lower() for l in t.splitlines() if l.strip().startswith("- ")}
    assert "make test" in agents, "AGENTS.md must mention the test command `make test`"
    assert "cents" in agents.lower(), "AGENTS.md must state that money is integer cents"
    assert any(l.strip() == "## Code Review Rules" for l in agents.splitlines()), "AGENTS.md needs a `## Code Review Rules` heading"
    assert not find_imports(agents), f"AGENTS.md must not contain @path imports (Codex reads them as text): {find_imports(agents)}"
    assert len(agents.splitlines()) <= 120, "keep AGENTS.md at 120 lines or fewer"
    first = next((l.strip() for l in claude.splitlines() if l.strip()), "")
    assert first == "@AGENTS.md", f"CLAUDE.md must start with the import `@AGENTS.md` (got {first!r}); prose like 'read AGENTS.md' is not an import"
    own = bullets(claude)
    assert own, "CLAUDE.md needs at least one Claude-only bullet"
    dup = own & bullets(agents)
    assert not dup, f"CLAUDE.md repeats AGENTS.md bullets (the copies will drift): {sorted(dup)[:2]}"
    assert "make test" not in claude, "CLAUDE.md repeats the test command; it already arrives through the import"


run_check("ex1", _check_ex1)

### Exercise 2 — Write a path-scoped testing rule

Return the full text of `.claude/rules/testing.md`. The repo keeps tests in three styles: `tests/**/test_*.py`, co-located `src/**/test_*.py`, and integration tests named `*_it.py` under `integration/`. The rule must:

- start with YAML frontmatter whose `paths` (list or comma-separated string) matches **all** of: `tests/test_api.py`, `tests/unit/test_money.py`, `src/shop/api/test_routes.py`, `integration/orders_it.py`
- match **none** of: `src/shop/pricing.py`, `docs/test_plan.md`, `src/shop/testing.py`, `integration/conftest.py`
- have a body with at least two bullet points

The checker parses the frontmatter and matches the globs the same way the practice notebook does. [Theory](README.md#42-path-scoped-conventions)

In [ ]:
def write_test_rule() -> str:
    # TODO: return the markdown text, frontmatter first
    raise NotImplementedError

In [ ]:
def _check_ex2():
    text = write_test_rule()
    assert isinstance(text, str) and text.startswith("---\n"), "the file must start with `---` frontmatter on the first line"
    meta, body = parse_frontmatter(text)
    globs = rule_globs(meta)
    assert globs, "frontmatter needs a `paths` field (list or comma-separated string)"
    must = ["tests/test_api.py", "tests/unit/test_money.py", "src/shop/api/test_routes.py", "integration/orders_it.py"]
    never = ["src/shop/pricing.py", "docs/test_plan.md", "src/shop/testing.py", "integration/conftest.py"]
    for f in must:
        assert any(glob_match(g, f) for g in globs), f"{f} should load the rule but no glob matches it ({globs})"
    for f in never:
        assert not any(glob_match(g, f) for g in globs), f"{f} should NOT load the rule ({globs}); check file extensions and prefixes"
    assert sum(l.strip().startswith("- ") for l in body.splitlines()) >= 2, "add at least two bullet points to the body"


run_check("ex2", _check_ex2)

### Exercise 3 — MCP config for both agents (team + personal)

Return a dict with three file texts:

- `".mcp.json"` (Claude Code, committed): a remote server `tickets` (`"type": "http"`) at `https://tickets.example.com/mcp`, with header `Authorization: Bearer <token>` where the token comes from the environment variable `TICKETS_TOKEN`. The URL may use a `${VAR:-default}` form as long as it expands to that address when the variable is unset.
- `".codex/config.toml"` (Codex, committed): `[mcp_servers.tickets]` with the same URL, the token from `TICKETS_TOKEN`, and `codex exec` must **fail** instead of running without it.
- `"~/.claude.json"` (personal, not committed): a **local-scope** stdio server `sandbox-db` (command `python`, args `["-m", "sandbox_db"]`) for the project at `/work/kestrel-shop` only.

No literal tokens anywhere; no variable that Claude Code blanks for remote servers; the personal server must not appear in the committed files. [Theory](README.md#44-mcp-servers-team-and-personal)

In [ ]:
def write_mcp_configs() -> dict[str, str]:
    # TODO: return {".mcp.json": ..., ".codex/config.toml": ..., "~/.claude.json": ...}
    raise NotImplementedError

In [ ]:
_BLANKED = {"ANTHROPIC_API_KEY", "ANTHROPIC_AUTH_TOKEN", "AWS_BEARER_TOKEN_BEDROCK", "NPM_TOKEN", "HTTPS_PROXY"}


def _expand_unset(s: str) -> str:
    """Expand ${VAR:-default} as if no variable were set (keeps ${VAR} without a default)."""
    return re.sub(r"\$\{([A-Za-z_]\w*):-([^}]*)\}", lambda m: m.group(2), s)


def _check_ex3():
    files = write_mcp_configs()
    assert {".mcp.json", ".codex/config.toml", "~/.claude.json"} <= set(files), "return all three files"
    team = json.loads(files[".mcp.json"]).get("mcpServers", {})
    t = team.get("tickets")
    assert t, ".mcp.json needs mcpServers.tickets"
    assert t.get("type") in {"http", "streamable-http"}, "tickets is a remote server: type http"
    assert _expand_unset(t.get("url", "")) == "https://tickets.example.com/mcp", "tickets url must be https://tickets.example.com/mcp"
    used = set(re.findall(r"\$\{([A-Za-z_]\w*)", files[".mcp.json"]))
    assert not used & _BLANKED, f"{used & _BLANKED} is read as empty toward remote servers; use your own variable name"
    auth = t.get("headers", {}).get("Authorization", "")
    assert re.fullmatch(r"Bearer \$\{TICKETS_TOKEN\}", auth), f"Authorization must be 'Bearer ${{TICKETS_TOKEN}}' (got {auth!r})"
    assert "sandbox-db" not in team, "the personal sandbox-db server must not be committed in .mcp.json"
    cx = tomllib.loads(files[".codex/config.toml"]).get("mcp_servers", {}).get("tickets", {})
    assert cx.get("url") == "https://tickets.example.com/mcp", "Codex: [mcp_servers.tickets] url"
    assert cx.get("bearer_token_env_var") == "TICKETS_TOKEN", "Codex: take the token from TICKETS_TOKEN with bearer_token_env_var (the Codex config documents no ${VAR} interpolation)"
    assert cx.get("required") is True, "Codex: required = true so codex exec fails when the server cannot start"
    assert "sandbox" not in files[".codex/config.toml"], "keep the personal server out of the committed Codex config"
    personal = json.loads(files["~/.claude.json"])
    local = personal.get("projects", {}).get("/work/kestrel-shop", {}).get("mcpServers", {})
    assert "sandbox-db" in local, "~/.claude.json: put sandbox-db under projects['/work/kestrel-shop'].mcpServers (local scope)"
    assert "sandbox-db" not in personal.get("mcpServers", {}), "top-level mcpServers is USER scope (all projects); the task asks for local scope"
    assert local["sandbox-db"].get("command") == "python" and local["sandbox-db"].get("args") == ["-m", "sandbox_db"], "sandbox-db: command python, args ['-m', 'sandbox_db']"
    blob = "".join(files.values())
    assert not re.search(r"Bearer\s+(?!\$\{)[A-Za-z0-9._-]{8,}", blob), "a literal bearer token is committed"


run_check("ex3", _check_ex3)

### Exercise 4 — Guardrails both agents enforce

Return `{".claude/settings.json": ..., ".codex/rules/team.rules": ...}`.

| Command / action | Claude Code must | Codex must |
|---|---|---|
| `python scripts/verify.py` (with or without arguments) | run without prompting (`allow`) | (no requirement) |
| any other `git push ...` | `ask` | `prompt` |
| `git push --force ...` and `git push -f ...` | `deny` | `forbidden` |
| `npm publish ...` | `deny` | `forbidden` |
| reading `.env` or `.env.<anything>` | `deny` | (no requirement) |

Every Codex rule must carry at least one `match` example, and all `match` / `not_match` examples must be correct. The checker evaluates your files with the same precedence rules as the practice notebook (Claude: deny → ask → allow; Codex: most restrictive wins). [Theory](README.md#45-permissions-sandbox-and-approvals)

In [ ]:
def write_guardrails() -> dict[str, str]:
    # TODO: return {".claude/settings.json": ..., ".codex/rules/team.rules": ...}
    raise NotImplementedError

In [ ]:
def _check_ex4():
    files = write_guardrails()
    s = json.loads(files[".claude/settings.json"])
    rules = load_codex_rules(files[".codex/rules/team.rules"])
    assert rules, "no prefix_rule(...) calls found in the rules file"
    expect_claude = {
        "python scripts/verify.py": "allow", "python scripts/verify.py --evidence evidence/verify.json": "allow",
        "git push": "ask", "git push origin feature/x": "ask",
        "git push --force origin main": "deny", "git push -f": "deny", "npm publish --access public": "deny",
    }
    for cmd, want in expect_claude.items():
        got = claude_decide("Bash", cmd, s)
        assert got == want, f"Claude: `{cmd}` -> {got}, expected {want} (deny is checked before ask, ask before allow)"
    for f in (".env", ".env.production"):
        assert claude_decide("Read", f, s) == "deny", f"Claude: reading {f} must be denied"
    expect_codex = {"git push origin feature/x": "prompt", "git push": "prompt", "git push --force origin main": "forbidden",
                    "git push -f": "forbidden", "npm publish": "forbidden", "git status": None}
    for cmd, want in expect_codex.items():
        got = codex_decide(cmd, rules)
        assert got == want, f"Codex: `{cmd}` -> {got}, expected {want}"
    for r in rules:
        assert r.get("match"), f"rule {r['pattern']} needs at least one match example"
        for ex in r.get("match", []):
            assert prefix_matches(r["pattern"], shlex.split(ex)), f"rule {r['pattern']}: match example {ex!r} does not match"
        for ex in r.get("not_match", []):
            assert not prefix_matches(r["pattern"], shlex.split(ex)), f"rule {r['pattern']}: not_match example {ex!r} matches"


run_check("ex4", _check_ex4)

### Exercise 5 — CI review commands and output parsing for both CLIs

1. `build_review_commands(prompt, schema, schema_path)` returns `{"claude": argv, "codex": argv}` (lists for `subprocess.run`, no shell):
   - **claude**: starts with `["claude", "-p", prompt]`; `--output-format json`; `--json-schema` with the schema as a JSON string; `--tools` naming only read-only tools (`Read`, `Grep`, `Glob`: `--tools` removes every other built-in tool, while `--allowedTools` only pre-approves and leaves `Edit`/`Write` available); `--permission-mode dontAsk` (anything that would prompt is denied); a `--max-turns` cap; no write-enabling flags (`--permission-mode` `acceptEdits`/`auto`/`bypassPermissions`, `--dangerously-skip-permissions`, `--allow-dangerously-skip-permissions`), and an `--allowedTools`, if you add one, must not name `Edit`, `Write` or unrestricted `Bash`.
   - **codex**: starts with `["codex", "exec"]`; `--json`; `--sandbox read-only`; `--output-schema schema_path`; prompt **last**; none of `--full-auto`, `--dangerously-bypass-approvals-and-sandbox`, `--yolo`, `danger-full-access`.
2. `parse_findings(tool, raw)` returns the list of findings:
   - `tool == "claude"`: `raw` is the single JSON object printed by `claude -p --output-format json`. Return `structured_output["findings"]`. Raise `AgentRunError` if `is_error` is true, `subtype` is not `"success"`, or `structured_output` is missing.
   - `tool == "codex"`: `raw` is JSONL from `codex exec --json`. The findings are in the **last** completed `agent_message` (its `text` is a JSON string). Raise `AgentRunError` on a `turn.failed` or top-level `error` event, or if no `turn.completed` arrived. Skip lines that are not JSON.

[Theory](README.md#47-ci-review-with-structured-output)

In [ ]:
def build_review_commands(prompt: str, schema: dict, schema_path: str) -> dict[str, list[str]]:
    # TODO
    raise NotImplementedError


def parse_findings(tool: str, raw: str) -> list[dict]:
    # TODO
    raise NotImplementedError

In [ ]:
_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["findings"],
           "properties": {"findings": {"type": "array", "items": {"type": "object"}}}}
_F1 = {"file": "a.py", "line": 3, "severity": "high", "issue": "x"}
_F2 = {"file": "b.py", "line": 9, "severity": "low", "issue": "y"}


def _flag(argv, *names):
    return next((argv[i + 1] for i, a in enumerate(argv[:-1]) if a in names), None)


def _expect_error(fn, why):
    try:
        fn()
    except AgentRunError:
        return
    except Exception as e:  # noqa: BLE001
        raise AssertionError(f"{why}: raise AgentRunError, not {type(e).__name__}")
    raise AssertionError(f"{why}: must raise AgentRunError (a failed run is not 'no findings')")


def _check_ex5():
    cmds = build_review_commands("Review the diff.", _SCHEMA, ".github/review.schema.json")
    c, x = cmds["claude"], cmds["codex"]
    assert all(isinstance(a, str) for a in c + x), "argv items must all be strings"
    assert c[:3] == ["claude", "-p", "Review the diff."], "claude argv must start with claude -p <prompt>"
    assert _flag(c, "--output-format") == "json", "claude: --output-format json"
    assert json.loads(_flag(c, "--json-schema") or "null") == _SCHEMA, "claude: --json-schema must carry the schema as JSON text"
    avail = {t for t in re.split(r"[\s,]+", _flag(c, "--tools") or "") if t}
    assert avail and avail <= {"Read", "Grep", "Glob"},         f"claude: restrict the AVAILABLE tools with --tools Read,Grep,Glob (got {sorted(avail) or 'no --tools'}); --allowedTools only pre-approves"
    pre = {t for t in re.split(r"[\s,]+", _flag(c, "--allowedTools", "--allowed-tools") or "") if t}
    assert not pre & {"Edit", "Write", "NotebookEdit", "Bash"}, f"claude: do not pre-approve write tools or unrestricted Bash (got {sorted(pre)})"
    modes = [c[i + 1] for i, a in enumerate(c[:-1]) if a == "--permission-mode"]
    assert modes and set(modes) == {"dontAsk"},         f"claude: exactly --permission-mode dontAsk (anything that would prompt is denied in CI); got {modes or 'none'}"
    risky = {"--dangerously-skip-permissions", "--allow-dangerously-skip-permissions"} & set(c)
    assert not risky, f"claude: remove {risky}"
    assert (_flag(c, "--max-turns") or "").isdigit(), "claude: cap the run with --max-turns N"
    assert x[:2] == ["codex", "exec"] and x[-1] == "Review the diff.", "codex: codex exec ... <prompt last>"
    assert "--json" in x, "codex: --json for JSONL events"
    assert _flag(x, "--sandbox", "-s") == "read-only", "codex: --sandbox read-only for a review"
    assert _flag(x, "--output-schema") == ".github/review.schema.json", "codex: --output-schema <schema_path>"
    bad = {"--full-auto", "--dangerously-bypass-approvals-and-sandbox", "--yolo", "danger-full-access"} & set(x)
    assert not bad, f"codex: remove {bad}"

    ok = json.dumps({"type": "result", "subtype": "success", "is_error": False, "result": "",
                     "structured_output": {"findings": [_F1]}})
    assert parse_findings("claude", ok) == [_F1], "claude: return structured_output['findings']"
    _expect_error(lambda: parse_findings("claude", json.dumps({"type": "result", "subtype": "error_max_turns", "is_error": True})),
                  "claude error_max_turns")
    _expect_error(lambda: parse_findings("claude", json.dumps({"type": "result", "subtype": "success", "is_error": False, "result": "ok"})),
                  "claude output without structured_output")
    stream = "\n".join([
        json.dumps({"type": "thread.started", "thread_id": "t1"}), "not json",
        json.dumps({"type": "item.completed", "item": {"type": "agent_message", "text": json.dumps({"findings": [_F1]})}}),
        json.dumps({"type": "item.completed", "item": {"type": "agent_message", "text": json.dumps({"findings": [_F2]})}}),
        json.dumps({"type": "turn.completed", "usage": {"output_tokens": 10}})])
    assert parse_findings("codex", stream) == [_F2], "codex: use the LAST completed agent_message and json.loads its text"
    _expect_error(lambda: parse_findings("codex", stream.replace('"turn.completed"', '"turn.failed"')), "codex turn.failed")
    _expect_error(lambda: parse_findings("codex", stream.rsplit("\n", 1)[0]), "codex stream without turn.completed")
    _expect_error(lambda: parse_findings("codex", stream + "\n" + json.dumps({"type": "error", "message": "boom"})), "codex top-level error event")


run_check("ex5", _check_ex5)

### Exercise 6 — Validate an evidence bundle against the verification contract

Implement `evidence_violations(evidence, contract, report)` returning a list of human-readable problems (empty = acceptable). `report` is the real output of the verify script (`exit_code`, `tests` = `{run, failed, errors, skipped}`, `test_ids`, `failing`). Rules:

1. Every name in `contract["evidence_fields"]` must be a key of `evidence`; if any is missing, return just the "missing" problems.
2. `evidence["status"]` must be `"verified"` or `"failed"`.
3. If `"verified"`: `report["exit_code"]` must be 0, `report` must show 0 failures and 0 errors, and `report["tests"]["run"]` must be at least `contract["baseline_tests_run"]`.
4. If `"failed"`: `evidence["failing_tests"]` must be non-empty.
5. Any ID in `contract["baseline_test_ids"]` that is missing from `report["test_ids"]` is a problem (a removed test), whatever the status.
6. `evidence["tests"]` must equal `report["tests"]` (the agent's numbers must match reality).
7. `evidence["attempts"]` must not exceed `contract["max_fix_attempts"]`.
8. `report["tests"]["skipped"]` must not exceed `contract["baseline_skipped"]`, whatever the status: a newly skipped test weakens the suite just like a deleted one.

[Theory](README.md#410-the-verification-contract-and-evidence)

In [ ]:
def evidence_violations(evidence: dict, contract: dict, report: dict) -> list[str]:
    # TODO
    raise NotImplementedError

In [ ]:
_CONTRACT = {"evidence_fields": ["task", "status", "files_changed", "commands", "tests", "tests_added", "failing_tests", "attempts"],
             "baseline_tests_run": 6, "baseline_test_ids": [f"t{i}" for i in range(6)], "max_fix_attempts": 2,
             "baseline_skipped": 0}
_GOOD_REPORT = {"exit_code": 0, "tests": {"run": 7, "failed": 0, "errors": 0, "skipped": 0},
                "test_ids": [f"t{i}" for i in range(7)], "failing": []}
_GOOD = {"task": "fix", "status": "verified", "files_changed": ["a.py"], "commands": [{"cmd": "verify", "exit_code": 0}],
         "tests": dict(_GOOD_REPORT["tests"]), "tests_added": ["t6"], "failing_tests": [], "attempts": 1}


def _check_ex6():
    assert evidence_violations(_GOOD, _CONTRACT, _GOOD_REPORT) == [], "an honest verified bundle must have no violations"
    fail_report = {"exit_code": 1, "tests": {"run": 7, "failed": 1, "errors": 0, "skipped": 0},
                   "test_ids": [f"t{i}" for i in range(7)], "failing": ["t6"]}
    honest_fail = {**_GOOD, "status": "failed", "tests": dict(fail_report["tests"]), "failing_tests": ["t6"], "attempts": 2}
    assert evidence_violations(honest_fail, _CONTRACT, fail_report) == [], "an honest 'failed' report is acceptable: reporting failure is allowed"
    cases = {
        "missing field": ({k: v for k, v in _GOOD.items() if k != "tests_added"}, _GOOD_REPORT),
        "verified on a failing run (agent's numbers copied from reality)": ({**_GOOD, "tests": dict(fail_report["tests"])}, fail_report),
        "verified with lying numbers": (_GOOD, fail_report),
        "fewer tests than baseline": ({**_GOOD, "tests": {"run": 5, "failed": 0, "errors": 0, "skipped": 0}},
                                      {"exit_code": 0, "tests": {"run": 5, "failed": 0, "errors": 0, "skipped": 0},
                                       "test_ids": ["t0", "t1", "t2", "t3", "t6"], "failing": []}),
        "a baseline test was replaced (same count)": (_GOOD, {**_GOOD_REPORT, "test_ids": ["t0", "t1", "t2", "t3", "t4", "t6", "t7"]}),
        "failed without naming tests": ({**honest_fail, "failing_tests": []}, fail_report),
        "too many attempts": ({**_GOOD, "attempts": 3}, _GOOD_REPORT),
        "unknown status": ({**_GOOD, "status": "done"}, _GOOD_REPORT),
        "a test newly skipped (same IDs and count, zero failures)": (
            {**_GOOD, "tests": {"run": 7, "failed": 0, "errors": 0, "skipped": 1}},
            {**_GOOD_REPORT, "tests": {"run": 7, "failed": 0, "errors": 0, "skipped": 1}}),
    }
    for label, (ev, rep) in cases.items():
        got = evidence_violations(ev, _CONTRACT, rep)
        assert isinstance(got, list) and got, f"case '{label}' must produce at least one violation"
    only = evidence_violations({k: v for k, v in _GOOD.items() if k != "tests_added"}, _CONTRACT, _GOOD_REPORT)
    assert len(only) == 1 and "tests_added" in only[0], "with missing fields, return only the missing-field problems, naming the field"


run_check("ex6", _check_ex6)

## Part C — Architecture scenario

**Fernbank Health** (fictional) has 120 engineers in 30 repositories. Sixty use Claude Code, sixty use Codex; both run in GitHub Actions. Constraints:

1. Patient-data services (`services/records/`, `services/billing/`) have stricter rules than the rest, including "never log request bodies".
2. Nobody's agent may push to `main`, run database migrations against production, or read `.env*` files, on any machine.
3. Every PR gets an AI review; the security team wants machine-readable findings in their dashboard and complains about noisy, repeated comments.
4. Contractors use unmanaged laptops and sometimes open PRs from forks.
5. Leadership wants proof that agent-made changes were verified before review.

Design the setup: where shared and tool-specific instructions live (and how you avoid duplication across 30 repos), how the stricter services get their rules in both tools, skills, MCP, which guardrails are enforced and how (laptop vs CI vs org-wide), the CI review and comment pipeline, and the verification contract. Name at least two trade-offs you accepted.

**Your answer:**

_(write here)_

<details><summary>Rubric: what a strong answer contains</summary>

- **Instruction layering:** `AGENTS.md` as the source of truth per repo; `CLAUDE.md` = `@AGENTS.md` + Claude-only notes; no `@` imports inside `AGENTS.md`. Shared org conventions distributed as a template or plugin rather than hand-copied (with a drift check in CI, like the practice audit).
- **Stricter services:** for Claude, a nested `CLAUDE.md` importing the service's guide (lazy load) or path-scoped rules with globs; for Codex, nested `AGENTS.md` only works when Codex starts in that directory, so add routing lines in the root `AGENTS.md` and/or start Codex in the service directory. Notes that "never log request bodies" also needs enforcement: a review rule plus a lint or CI check.
- **Skills:** shared skills once in `.agents/skills/`, symlinked into `.claude/skills/` (Windows fallback: generated copies + drift check); spec-only frontmatter; side-effecting skills have `disable-model-invocation` (Claude) / `allow_implicit_invocation: false` (Codex).
- **MCP:** team servers in `.mcp.json` / `.codex/config.toml` with env credentials (own variable names; not Claude's own credential variables); personal servers in `~/.claude.json` / `~/.codex/config.toml`; `required = true` where CI depends on a server; Codex project config needs trust.
- **Enforcement:** the "never" list as managed-settings `deny` rules (Claude) and admin `requirements.toml` `forbidden` rules (Codex), which also bind contractors' sessions only if their machines are managed, so CI-side controls (branch protection, required reviews, no prod credentials in CI) carry the guarantee for unmanaged laptops. Sandboxes on: Claude `sandbox.enabled`, Codex `workspace-write` + `on-request`, network off by default.
- **CI review:** both agents with one strict schema (`--json-schema` / `--output-schema`); key-holding jobs read-only; a separate posting job without model keys, running base-branch code; diff/evidence/duplicate filters; previous findings passed on re-review; per-file + integration passes for big PRs. Fork PRs: no secrets (GitHub withholds them), so review runs only on same-repo branches or after a maintainer label; untrusted PR config (hooks, `.mcp.json`) never runs with secrets (`--bare` for the Claude CLI path).
- **Verification:** contract in `AGENTS.md` + machine-readable JSON; verify script writes evidence; a validator in CI cross-checks the agent's claims against the real run (baseline count, no removed/skipped tests); failure policy (stop after N attempts, report `failed`).
- **Trade-offs named**, e.g. routing tables are guidance (Codex may skip them) vs nested files that need the right working directory; symlinks vs Windows; two reviewers cost more but cross-check each other; strict filters may drop a true finding with a wrong line number.
</details>

## Scorecard

In [ ]:
quiz_correct = QUIZ_SCORE["correct"]
ex_passed = sum(RESULTS.values())
total = QUIZ_SCORE["total"] + len(RESULTS)
combined = (quiz_correct + ex_passed) / total
print(f"Quiz:      {quiz_correct}/{QUIZ_SCORE['total']}")
print(f"Exercises: {ex_passed}/{len(RESULTS)}  {RESULTS}")
print(f"Combined:  {combined:.0%}  (pass mark 72%)")
print("PASS ✅" if combined >= 0.72 else "RETRY 🔁 — reread the README sections you missed, then try again")

---
## Solutions (spoilers)

Try everything first. Each block is collapsed.

<details><summary>Quiz answers with explanations</summary>

- **Q1** **C.** Once any `CLAUDE.md`-family file exists (and `CLAUDE.local.md` counts), Claude Code reads those instead of `AGENTS.md`. A committed `CLAUDE.md` with `@AGENTS.md` brings the shared file back for everyone, in every session type. B: Claude Code ignores that setting in project and local settings files. A leaks personal data into the shared file. D breaks Codex.

- **Q2** **A.** Codex walks from the project root **down to** the working directory and never reads directories below it. Start it where the work is, or route from the root file. There is no cache to clear (B). `AGENTS.override.md` is optional at any level (C). Nothing in the scenario points at the size cap (D), and the cap would drop later files, not explain this pattern.

- **Q3** **D.** Codex scans `.agents/skills/`, Claude Code scans `.claude/skills/`, and both follow symlinked skill folders. Codex's implicit-invocation policy lives in `agents/openai.yaml`, so the shared `SKILL.md` keeps spec-only frontmatter. A duplicates the skill into a folder Codex does not scan. B loads in every session and is not a skill. C is per-laptop, not shared.

- **Q4** **B.** In a remote server's `url` and `headers`, Claude Code reads its own credentials (such as `ANTHROPIC_API_KEY`) as empty, even with a default, so the server receives `Bearer `. Use a separate variable name. A is false: `headers` is an expansion location. C: defaults are ignored for these variables. D: scope does not change this protection.

- **Q5** **C.** Claude Code checks deny, then ask, then allow; the first match wins, and an allow rule can never carve an exception out of a deny rule from any scope. A, B and D describe precedence rules that do not exist.

- **Q6** **D.** Codex loads project `.codex/` layers (config, rules, hooks) only for trusted projects, and trust is recorded per path in the user config (`projects.<path>.trust_level` in `~/.codex/config.toml`), which an ephemeral runner starts without. The docs do not spell out how `codex exec` treats a path with no trust entry, but D is the only option consistent with the symptom, and the fix (configure CI explicitly instead of relying on the project layer) is right either way. A and B are false; `required = true` exists exactly to make startup fail when a server cannot initialize. C is false.

- **Q7** **A.** Deterministic filters beat instructions: GitHub accepts inline comments only on diff lines, and an evidence string that is not on the cited line exposes an unsupported finding. B, C and D ask the model to try harder without any check.

- **Q8** **B.** The job that holds the model key cannot write to the repository, and the job that writes never sees the key. A combines both. C removes the sandbox and still mixes key and write access. D puts a secret in the repository.

- **Q9** **D.** Several valid designs with infrastructure consequences across multiple areas is the case for plan mode. A, B and C are small, well-understood changes for direct execution.

- **Q10** **C.** The contract makes 'done' checkable: test count never below the baseline and no removed or skipped tests. Deleting failing tests is the classic way to fake a green run. A and B accept it; D hides the agent's behavior instead of reporting it.
</details>

<details><summary>Exercise 1 — write_instruction_layer</summary>

```python
def write_instruction_layer() -> dict[str, str]:
    agents = """# AGENTS.md

## Project
- Python 3.11+. Run `make test` before you finish a task.
- Money is always integer cents; never use float for prices.
- Read configuration from environment variables; never commit secrets.

## Code Review Rules
- Flag float arithmetic on money. Safe path: integer cents.
"""
    claude = """@AGENTS.md

## Claude Code only
- Use plan mode for changes that touch more than 3 files.
- Run `/codebase-map` before exploring an unfamiliar package.
"""
    return {"AGENTS.md": agents, "CLAUDE.md": claude}
```
</details>

<details><summary>Exercise 2 — write_test_rule</summary>

```python
def write_test_rule() -> str:
    return """---
paths:
  - "**/test_*.py"
  - "integration/**/*_it.py"
---
# Testing conventions
- One behavior per test method, named `test_<behavior>`.
- Every bug fix adds a regression test that fails before the fix.
"""
```
</details>

<details><summary>Exercise 3 — write_mcp_configs</summary>

```python
def write_mcp_configs() -> dict[str, str]:
    mcp_json = json.dumps({"mcpServers": {"tickets": {
        "type": "http",
        "url": "https://tickets.example.com/mcp",
        "headers": {"Authorization": "Bearer ${TICKETS_TOKEN}"},
    }}}, indent=2)
    codex_toml = """
[mcp_servers.tickets]
url = "https://tickets.example.com/mcp"
bearer_token_env_var = "TICKETS_TOKEN"
required = true
"""
    personal = json.dumps({"projects": {"/work/kestrel-shop": {"mcpServers": {
        "sandbox-db": {"type": "stdio", "command": "python", "args": ["-m", "sandbox_db"]}}}}}, indent=2)
    return {".mcp.json": mcp_json, ".codex/config.toml": codex_toml, "~/.claude.json": personal}
```
</details>

<details><summary>Exercise 4 — write_guardrails</summary>

```python
def write_guardrails() -> dict[str, str]:
    settings = json.dumps({"permissions": {
        "allow": ["Bash(python scripts/verify.py *)"],
        "ask": ["Bash(git push *)"],
        "deny": ["Bash(git push --force *)", "Bash(git push -f *)", "Bash(npm publish *)",
                 "Read(./.env)", "Read(./.env.*)"],
    }}, indent=2)
    rules = """
prefix_rule(pattern=["git", "push"], decision="prompt",
            justification="Pushing needs a human.", match=["git push origin main"], not_match=["git status"])
prefix_rule(pattern=["git", "push", ["--force", "-f"]], decision="forbidden",
            justification="No force-push.", match=["git push --force", "git push -f origin x"],
            not_match=["git push origin main"])
prefix_rule(pattern=["npm", "publish"], decision="forbidden",
            justification="Releases go through CI.", match=["npm publish --access public"])
"""
    return {".claude/settings.json": settings, ".codex/rules/team.rules": rules}
```
</details>

<details><summary>Exercise 5 — build_review_commands + parse_findings</summary>

```python
def build_review_commands(prompt: str, schema: dict, schema_path: str) -> dict[str, list[str]]:
    claude = ["claude", "-p", prompt, "--output-format", "json", "--json-schema", json.dumps(schema),
              "--tools", "Read,Grep,Glob", "--permission-mode", "dontAsk", "--max-turns", "8"]
    codex = ["codex", "exec", "--json", "--sandbox", "read-only", "--output-schema", schema_path, prompt]
    return {"claude": claude, "codex": codex}


def parse_findings(tool: str, raw: str) -> list[dict]:
    if tool == "claude":
        out = json.loads(raw)
        if out.get("is_error") or out.get("subtype") != "success" or "structured_output" not in out:
            raise AgentRunError(f"claude run failed: {out.get('subtype')}")
        return out["structured_output"]["findings"]
    if tool == "codex":
        final, completed = None, False
        for line in raw.splitlines():
            try:
                ev = json.loads(line)
            except json.JSONDecodeError:
                continue
            if ev.get("type") == "item.completed" and ev.get("item", {}).get("type") == "agent_message":
                final = ev["item"]["text"]
            elif ev.get("type") == "turn.completed":
                completed = True
            elif ev.get("type") in {"turn.failed", "error"}:
                raise AgentRunError(f"codex run failed: {ev}")
        if not completed or final is None:
            raise AgentRunError("codex run did not complete")
        return json.loads(final)["findings"]
    raise ValueError(f"unknown tool {tool!r}")
```
</details>

<details><summary>Exercise 6 — evidence_violations</summary>

```python
def evidence_violations(evidence: dict, contract: dict, report: dict) -> list[str]:
    missing = [f"missing field: {k}" for k in contract["evidence_fields"] if k not in evidence]
    if missing:
        return missing
    problems = []
    status = evidence["status"]
    if status == "verified":
        if report["exit_code"] != 0:
            problems.append("verified, but the verify command failed")
        if report["tests"]["failed"] or report["tests"]["errors"]:
            problems.append("verified, but tests fail")
        if report["tests"]["run"] < contract["baseline_tests_run"]:
            problems.append("fewer tests ran than the baseline")
    elif status == "failed":
        if not evidence["failing_tests"]:
            problems.append("failed status must list the failing tests")
    else:
        problems.append(f"unknown status {status!r}")
    removed = [t for t in contract["baseline_test_ids"] if t not in report["test_ids"]]
    if removed:
        problems.append(f"tests removed: {removed}")
    if evidence["tests"] != report["tests"]:
        problems.append("reported test counts do not match the verify report")
    if evidence["attempts"] > contract["max_fix_attempts"]:
        problems.append("too many fix attempts")
    if report["tests"]["skipped"] > contract["baseline_skipped"]:
        problems.append("tests newly skipped")
    return problems
```
</details>

## Recap

- One source of truth (`AGENTS.md`, imported by `CLAUDE.md`), file-type guides loaded by globs for Claude and by routing for Codex, and shared skills in `.agents/skills/` symlinked for Claude.
- Credentials from the environment with your own variable names; personal MCP servers stay in your home directory.
- Enforcement lives in `deny` rules, `.rules` files, sandboxes and managed settings, not in instruction files.
- CI review: one strict schema for both CLIs, failures raised (never read as "no findings"), and evidence checked against the real verify run.

Next: Capstone 5 — Enterprise RAG assistant with evals and observability (planned) · Back to the [capstone README](README.md)